# 15 · PolyShrink: the full method (new)

**What the runs so far showed** (Kaggle T4, `experimented/`): running a consecutive pair **in parallel** keeps the
most few-shot ability of any method (avg acc 33.7 at 2 layers, 28.0 at 4, vs drop 30.6 / 23.8). PRISM-Shrink (anneal
that parallel union down to one layer) gives the best perplexity at the same size (+18.8 % vs +27.5 % for AAM and
+110 % for drop at 4 layers), but its few-shot accuracy only matches drop. Two failure modes remain, and each new
notebook attacks one:

| Failure mode | Evidence | Fix | Notebook |
|---|---|---|---|
| The merge is trained on short plain text, so the long-range copying few-shot prompts need is never exercised | perplexity improves while few-shot falls, for every closed-form and short-distillation merge | **ICL-preserving packed distillation**: `[a, b, a]` sequences (anchor language, another language, the anchor again) | 13 |
| One shared layer must serve 6 languages; what merging loses is language-specific | the worst language loses 1.5-2x the average perplexity | **Language-specific residual (LSR)**: shared merged layer + per-language rank-16 delta, routed by a frozen linear language probe | 14 |
| Which pairs: angular distance picks pairs that change the state least, not pairs that are most parallel | parallel-union error differs between pairs with similar angular distance | **CoPlan**: merge the pairs whose exact parallel union has the lowest error | 15 (full method) |

Notebook 15 is the full method (**PolyShrink = CoPlan + PRISM-Shrink + ICL packing + LSR**); 11 (archived), 13 and 14
are its ablations, so the paper can attribute every gain.

**This notebook: everything together.**
1. **CoPlan** picks the pairs: the exact parallel union of every adjacent pair is built and its held-out error
   measured in the original model; the most parallelizable non-overlapping pairs are merged (printed next to the
   angular-distance plan).
2. **PRISM-Shrink**: exact parallel union, discarded parts annealed out under distillation, standard layer extracted.
3. **ICL-preserving packed distillation** (as in 13).
4. **Language-specific residual** (as in 14).

Because CoPlan may choose different pairs, the `drop` reference row is recomputed for those pairs (same table).
**~20–25 min training + ~10 min evaluation per merge count on a T4, + ~2 min planning.**

In [ ]:
# Colab: Runtime -> Change runtime type -> T4 GPU.  Kaggle: Settings -> Accelerator GPU T4/P100, Internet ON.
!pip -q install -U transformers datasets pandas matplotlib "pyarrow>=19,<24"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Settings
Keep the same settings (and `data_dir`) in every notebook, so data, reference rows and results are shared and the final table compares all algorithms.

In [ ]:
import os
CFG = dict(
    model="Qwen/Qwen3-0.6B-Base",      # any Qwen3 base model, e.g. "Qwen/Qwen3-1.7B-Base" (needs more GPU memory)
    # data + results folder, shared by all notebooks. Colab: Google Drive (mounted when data loads), so the original /
    # drop reference rows are computed ONCE and every algorithm lands in one table. Kaggle: /kaggle/working.
    data_dir=("/kaggle/working/layer_merging" if os.path.exists("/kaggle/working") else
              "/content/drive/MyDrive/layer_merging" if os.path.exists("/content") else "data"),
    langs=["en", "hi", "fr", "es", "de", "zh"],   # also available: "ja", "ru"
    protect=2,                         # never merge the first / last 2 layers
    # calibration / training text: Wiki-40B train
    seq=256, n_train=300, n_calib=16,  # sequences per language (the last calibration sequence is held out)
    # test sets, per language (few-shot everywhere)
    n_mc4=16, n_wiki=16,               # perplexity sequences
    n_mmmlu=100, n_include=100, n_mgsm=40, n_bele=100, n_xnli=100,
    shots=dict(mmmlu=5, include=5, mgsm=4, belebele=3, xnli=3),
    max_len=2048, mgsm_tokens=160,     # prompt length cap (oldest shots are cut), MGSM answer length
    ridge=1e-2,                        # ridge strength of all least-squares fits
    merge_counts=[2, 4],               # PRISM notebooks: 2 and 4 layers removed (6 is near chance for every method); add 6 if time allows
    sink_aware=True,                   # fit every token by RELATIVE error, incl. the position-0 attention sink (False = ablation)
    refit=True,                        # closed-form o_proj / down_proj refit after selection (False = ablation)
    steps=400,                         # training steps (each: one sequence per language)
    lr=5e-5,
    anneal_frac=0.7,                   # fraction of steps over which ALPHA goes 1 -> 0; 0 = no annealing (ablation)
    hid_w=1.0,                         # hidden-state loss weight next to the KL loss
    select="energy",                     # PRISM start: "energy" | "energy" | "polyglot"
    lsr_rank=16,                       # rank of the language-specific residual (notebooks 14, 15)
)
if os.environ.get("SMOKE"):  # tiny local test
    CFG.update(data_dir=os.environ["SMOKE"], seq=64, n_train=8, n_calib=3, n_mc4=2, n_wiki=2, n_mmmlu=4, n_include=4,
               n_mgsm=2, n_bele=4, n_xnli=3, mgsm_tokens=24, merge_counts=[1, 2], steps=4, n_search=2)

## Data
Calibration/training: Wiki-40B train. Tests (never used for fitting): **MMMLU** (English: MMLU) 5-shot, **INCLUDE** 5-shot, **MGSM** 4-shot chain of thought, **Belebele** 3-shot, **XNLI** 3-shot, plus mC4 and Wiki-40B test perplexity. Few-shot examples come from dev/validation/train splits, or from rows held out of evaluation.

In [ ]:
import ast, re, json, math, time, copy, gc, glob, random, hashlib, contextlib
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    display
except NameError:
    display = print
torch.manual_seed(0)
DEV = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
LANGS, SLUG = CFG["langs"], CFG["model"].split("/")[-1]
# wiki40b, belebele, mC4, MMMLU (en = original MMLU), INCLUDE, MGSM, XNLI.  None = the benchmark lacks that language.
LCODE = {"en": ("en", "eng_Latn", "en", "EN", None, "en", "en"),
         "hi": ("hi", "hin_Deva", "hi", "HI_IN", "Hindi", None, "hi"),
         "fr": ("fr", "fra_Latn", "fr", "FR_FR", "French", "fr", "fr"),
         "es": ("es", "spa_Latn", "es", "ES_LA", "Spanish", "es", "es"),
         "de": ("de", "deu_Latn", "de", "DE_DE", "German", "de", "de"),
         "zh": ("zh-cn", "zho_Hans", "zh", "ZH_CN", "Chinese", "zh", "zh"),
         "ja": ("ja", "jpn_Jpan", "ja", "JA_JP", "Japanese", "ja", None),
         "ru": ("ru", "rus_Cyrl", "ru", None, "Russian", "ru", "ru")}
XNLI_WORDS = {"en": ("right", "Yes", "Also", "No"), "fr": ("correct", "Oui", "Aussi", "Non"),
              "es": ("correcto", "Sí", "Asi que", "No"), "de": ("richtig", "Ja", "Auch", "Nein"),
              "hi": ("सही", "हाँ", "इसलिए", "नहीं"), "zh": ("正确", "是的", "所以", "不是"),
              "ru": ("Правильно", "Да", "Так", "Нет")}
print("device:", DEV)


def wiki40b_text(t):
    if t[:2] in ("b'", 'b"'):  # Wiki-40B stores text as an escaped bytes literal
        t = ast.literal_eval(t).decode("utf-8")
    t = re.sub(r"_START_(ARTICLE|SECTION)_\n[^\n]*\n", "", t)
    return t.replace("_START_PARAGRAPH_\n", "").replace("_NEWLINE_", "\n").strip()


def token_chunks(texts, tok, seq, n):
    """Pack documents (EOS-separated, <= 2*seq tokens each) into n chunks of seq tokens."""
    buf, out = [], []
    for t in texts:
        buf += tok(t[:20000])["input_ids"][:2 * seq] + [tok.eos_token_id]
        while len(buf) >= seq and len(out) < n:
            out.append(buf[:seq])
            buf = buf[seq:]
        if len(out) == n:
            return torch.tensor(out)
    raise RuntimeError(f"only {len(out)}/{n} chunks available")


def optional(what, fn):
    try:
        return fn()
    except Exception as e:
        print(f"  WARNING: could not load {what} ({type(e).__name__}: {e}); continuing without it")
        return None


def load_bench(l):
    """Few-shot benchmarks: {name: {"shots": [...], "items": [...]}}. Shots never overlap the evaluated items."""
    from datasets import load_dataset
    w, b, m, mm, inc, mg, xn = LCODE[l]
    c, k, out = CFG, CFG["shots"], {}
    rnd = lambda: random.Random(0)  # fresh per benchmark: same Belebele order in every language
    mc = lambda q, opts, ans: dict(q=q, opts=[str(o) for o in opts], ans=int(ans))
    if mm == "EN":
        def f():
            dev = [mc(r["question"], r["choices"], r["answer"]) for r in load_dataset("cais/mmlu", "all", split="dev")]
            test = [mc(r["question"], r["choices"], r["answer"]) for r in load_dataset("cais/mmlu", "all", split="test")]
            rnd().shuffle(dev), rnd().shuffle(test)
            return dict(shots=dev[:k["mmmlu"]], items=test[:c["n_mmmlu"]])
        out["mmmlu"] = optional("MMLU (en)", f)
    elif mm:
        def f():
            rows = [mc(r["Question"], [r[x] for x in "ABCD"], "ABCD".index(str(r["Answer"]).strip()))
                    for r in load_dataset("openai/MMMLU", mm, split="test")]
            rnd().shuffle(rows)  # MMMLU has no dev split: the first rows become the shots and are not evaluated
            return dict(shots=rows[:k["mmmlu"]], items=rows[k["mmmlu"]:k["mmmlu"] + c["n_mmmlu"]])
        out["mmmlu"] = optional(f"MMMLU ({mm})", f)
    if inc:
        def f():
            row = lambda r: mc(r["question"], [r[f"option_{x}"] for x in "abcd"], r["answer"])
            test = [row(r) for r in load_dataset("CohereLabs/include-base-44", inc, split="test")]
            rnd().shuffle(test)
            shots = [row(r) for r in load_dataset("CohereLabs/include-base-44", inc, split="validation")][:k["include"]]
            return dict(shots=shots, items=test[:c["n_include"]])
        out["include"] = optional(f"INCLUDE ({inc})", f)
    if mg:
        out["mgsm"] = optional(f"MGSM ({mg})", lambda: dict(
            shots=[dict(r) for r in load_dataset("juletxara/mgsm", mg, split="train")][:k["mgsm"]],
            items=[dict(question=r["question"], answer=str(r["answer_number"])) for r in
                   load_dataset("juletxara/mgsm", mg, split="test")][:c["n_mgsm"]]))
    def f():
        rows = [dict(x) for x in load_dataset("facebook/belebele", b, split="test")]
        rnd().shuffle(rows)
        return dict(shots=rows[-k["belebele"]:], items=rows[:c["n_bele"]])
    out["belebele"] = optional(f"Belebele ({b})", f)
    if xn:
        def f():
            val = [dict(x) for x in load_dataset("facebook/xnli", xn, split="validation")]
            shots = [next(x for x in val if x["label"] == lab) for lab in (0, 2, 1)][:k["xnli"]]
            test = [dict(x) for x in load_dataset("facebook/xnli", xn, split="test")]
            rnd().shuffle(test)
            return dict(shots=shots, items=test[:c["n_xnli"]])
        out["xnli"] = optional(f"XNLI ({xn})", f)
    return {n: v for n, v in out.items() if v}


def load_data(tok):
    c = CFG
    if c["data_dir"].startswith("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    key = json.dumps([SLUG, LANGS] + [c[x] for x in ("seq", "n_train", "n_calib", "n_mc4", "n_wiki", "n_mmmlu",
                                                     "n_include", "n_mgsm", "n_bele", "n_xnli", "shots")], sort_keys=True)
    path = f"{c['data_dir']}/data_{SLUG}_{hashlib.md5(key.encode()).hexdigest()[:8]}.pt"
    if os.path.exists(path):
        print("using cached data", path)
        return torch.load(path, weights_only=False)
    print("Downloading data (first run only, ~5-15 min). Tip: add HF_TOKEN as a Colab/Kaggle secret to avoid rate limits.")
    from datasets import load_dataset
    d = {}
    for l in LANGS:
        w, _, m, *_ = LCODE[l]
        t0 = time.time()
        train = token_chunks((wiki40b_text(x["text"]) for x in load_dataset("google/wiki40b", w, split="train", streaming=True)),
                             tok, c["seq"], c["n_train"] + c["n_calib"])
        d[l] = {"calib": train[:c["n_calib"]], "train": train[c["n_calib"]:],
                "mc4": token_chunks((x["text"] for x in load_dataset("allenai/c4", m, split="validation", streaming=True)),
                                    tok, c["seq"], c["n_mc4"]),
                "wiki": optional(f"Wiki-40B test ({w})", lambda: token_chunks(
                    (wiki40b_text(x["text"]) for x in load_dataset("google/wiki40b", w, split="test", streaming=True)),
                    tok, c["seq"], c["n_wiki"])),
                "bench": load_bench(l)}
        sizes = ", ".join(f"{n} {len(v['items'])}" for n, v in d[l]["bench"].items())
        print(f"  {l}: {sizes} ({time.time() - t0:.0f}s)", flush=True)
    os.makedirs(c["data_dir"], exist_ok=True)
    torch.save(d, path)
    return d


tok = AutoTokenizer.from_pretrained(CFG["model"])
tok.truncation_side = "left"  # over-long few-shot prompts lose their oldest shots, never the question
DATA = load_data(tok)

## Few-shot evaluation

In [ ]:
AMP_DTYPE = (torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16) if DEV == "cuda" else None


def amp():
    return torch.autocast("cuda", dtype=AMP_DTYPE) if AMP_DTYPE else contextlib.nullcontext()


@torch.no_grad()
def token_logprobs(model, ids, mask=None, chunk=1024):
    with amp():
        h = model.model(input_ids=ids, attention_mask=mask).last_hidden_state[:, :-1]
    tgt = ids[:, 1:]
    lp = [-F.cross_entropy(model.lm_head(hc.float()).float(), tc, reduction="none")
          for hc, tc in zip(h.reshape(-1, h.shape[-1]).split(chunk), tgt.reshape(-1).split(chunk))]
    return torch.cat(lp).view_as(tgt)


@torch.no_grad()
def ppl(model, x, bs=4):
    nll, n = 0.0, 0
    for b in x.split(bs):
        b = b.to(DEV)
        nll -= token_logprobs(model, b).sum().item()
        n += b[:, 1:].numel()
    return math.exp(nll / n)


@torch.no_grad()
def letter_acc(model, prompts, answers, bs=4):
    """Pick A/B/C/D by the next-token logit after the prompt."""
    ids = [tok.encode(x)[0] for x in (" A", " B", " C", " D")]
    tok.padding_side, correct = "left", 0
    for i in range(0, len(prompts), bs):
        enc = tok(prompts[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=CFG["max_len"]).to(DEV)
        with amp():
            last = model.model(**enc).last_hidden_state[:, -1]
        pred = model.lm_head(last.float())[:, ids].argmax(-1).cpu()
        correct += (pred == torch.tensor(answers[i:i + bs])).sum().item()
    return correct / len(prompts)


def mc_text(x):
    return f"Question: {x['q'].strip()}\n" + "".join(f"{a}. {o.strip()}\n" for a, o in zip("ABCD", x["opts"])) + "Answer:"


def mc_eval(model, B, lang):  # MMMLU / MMLU / INCLUDE, k-shot
    head = "The following are multiple choice questions (with answers).\n\n"
    head += "".join(f"{mc_text(s)} {'ABCD'[s['ans']]}\n\n" for s in B["shots"])
    return letter_acc(model, [head + mc_text(x) for x in B["items"]], [x["ans"] for x in B["items"]])


def bele_text(x):
    return (f"P: {x['flores_passage']}\nQ: {x['question']}\nA: {x['mc_answer1']}\nB: {x['mc_answer2']}\n"
            f"C: {x['mc_answer3']}\nD: {x['mc_answer4']}\nAnswer:")


def bele_eval(model, B, lang):  # Belebele, k-shot
    head = "".join(f"{bele_text(s)} {'ABCD'[int(s['correct_answer_num']) - 1]}\n\n" for s in B["shots"])
    return letter_acc(model, [head + bele_text(x) for x in B["items"]], [int(x["correct_answer_num"]) - 1 for x in B["items"]])


@torch.no_grad()
def xnli_eval(model, B, lang, bs=4):  # XNLI, k-shot; picks the most likely of the three completed sentences
    q, *words = XNLI_WORDS[lang]
    head = "".join(f"{s['premise']}, {q}? {words[s['label']]}, {s['hypothesis']}\n" for s in B["shots"])
    tok.padding_side, correct = "right", 0
    for i in range(0, len(B["items"]), bs):
        batch = B["items"][i:i + bs]
        enc = tok([f"{head}{x['premise']}, {q}? {w}, {x['hypothesis']}" for x in batch for w in words],
                  return_tensors="pt", padding=True, truncation=True, max_length=CFG["max_len"]).to(DEV)
        tl = token_logprobs(model, enc["input_ids"], enc["attention_mask"])
        pred = (tl * enc["attention_mask"][:, 1:]).sum(-1).cpu().view(len(batch), 3).argmax(-1)
        correct += (pred == torch.tensor([x["label"] for x in batch])).sum().item()
    return correct / len(B["items"])


def prefix(s):
    """'Frage: ...' -> 'Frage:' (MGSM shots carry the question/answer words of their language)."""
    m = re.match(r"^[^:：]{1,40}[:：]", s)
    return m.group(0) if m else ""


@torch.no_grad()
def mgsm_eval(model, B, lang, bs=8):  # MGSM, k-shot chain of thought, greedy generation, exact final number
    qp, ap = prefix(B["shots"][0]["question"]), prefix(B["shots"][0]["answer"])
    head = "".join(f"{s['question'].strip()}\n{s['answer'].strip()}\n\n" for s in B["shots"])
    tok.padding_side, correct = "left", 0
    for i in range(0, len(B["items"]), bs):
        batch = B["items"][i:i + bs]
        enc = tok([f"{head}{qp} {x['question'].strip()}\n{ap}" for x in batch], return_tensors="pt", padding=True,
                  truncation=True, max_length=CFG["max_len"]).to(DEV)
        with amp():
            gen = model.generate(**enc, max_new_tokens=CFG["mgsm_tokens"], do_sample=False, use_cache=True,
                                 stop_strings=["\n\n", qp] if qp else ["\n\n"], tokenizer=tok, pad_token_id=tok.pad_token_id)
        for x, g in zip(batch, tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)):
            nums = re.findall(r"-?\d[\d,]*(?:\.\d+)?", g.split("\n\n")[0])
            correct += bool(nums) and abs(float(nums[-1].replace(",", "")) - float(x["answer"].replace(",", ""))) < 1e-6
    return correct / len(B["items"])


TASKS = {"mmmlu": mc_eval, "include": mc_eval, "mgsm": mgsm_eval, "belebele": bele_eval, "xnli": xnli_eval}


def evaluate(model):
    model.eval()
    r = {n: {} for n in ["mc4", "wiki", *TASKS]}
    for l in LANGS:
        t0, d = time.time(), DATA[l]
        r["mc4"][l] = round(ppl(model, d["mc4"]), 3)
        if d["wiki"] is not None:
            r["wiki"][l] = round(ppl(model, d["wiki"]), 3)
        for n, fn in TASKS.items():
            if n in d["bench"]:
                r[n][l] = round(fn(model, d["bench"][n], l), 4)
        print(f"    eval {l}: {time.time() - t0:.0f}s", flush=True)
    return r

## Original model, calibration states, planner

In [ ]:
teacher = AutoModelForCausalLM.from_pretrained(CFG["model"], dtype=torch.float32).to(DEV).eval()
teacher.config.use_cache = False
teacher.requires_grad_(False)
L = teacher.config.num_hidden_layers
if AMP_DTYPE:  # half-precision eval is ~4x faster on T4; switch it off if it changes the original model's perplexity
    x = DATA[LANGS[0]]["mc4"][:2]
    p16 = ppl(teacher, x)
    _a, AMP_DTYPE = AMP_DTYPE, None
    p32 = ppl(teacher, x)
    AMP_DTYPE = _a if abs(p16 / p32 - 1) < 0.02 else None
    print(f"mixed precision {AMP_DTYPE} (ppl fp32 {p32:.3f} vs half {p16:.3f})")


def flat(t):
    t = t[0] if isinstance(t, tuple) else t
    return t[:, 1:].reshape(-1, t.shape[-1]).float().cpu()  # position 0 (attention sink) excluded everywhere


@torch.no_grad()
def capture(model, li, inputs=(), outputs=()):
    """Inputs / outputs of submodules of layer li ("" = the layer itself) on the current calibration tokens of all
    languages. Output keys get a ':out' suffix."""
    layer, got, hooks = model.model.layers[li], {}, []
    for n in inputs:
        hooks.append(layer.get_submodule(n).register_forward_pre_hook(lambda m, a, n=n: got.setdefault(n, []).append(flat(a[0]))))
    for n in outputs:
        hooks.append(layer.get_submodule(n).register_forward_hook(lambda m, a, o, n=n: got.setdefault(n + ":out", []).append(flat(o))))
    try:
        for l in LANGS:
            with amp():
                model.model(DATA[l]["calib"][:NCAL].to(DEV))
    finally:
        for h in hooks:
            h.remove()
    return {k: torch.cat(v) for k, v in got.items()}


@torch.no_grad()
def layer_inputs():
    got, hooks = {i: [] for i in range(L)}, []
    for i in range(L):
        hooks.append(teacher.model.layers[i].register_forward_pre_hook(lambda m, a, i=i: got[i].append(flat(a[0]))))
    for l in LANGS:
        teacher.model(DATA[l]["calib"].to(DEV))  # fp32: these are the fitting targets
    for h in hooks:
        h.remove()
    return [torch.cat(got[i]) for i in range(L)]


n_cal, S = DATA[LANGS[0]]["calib"].shape
HS = layer_inputs()  # HS[i] = original hidden state entering layer i, [calibration tokens, d]
SEQ_ID = torch.arange(n_cal).repeat_interleave(S - 1).repeat(len(LANGS))
TOK_LANG = torch.arange(len(LANGS)).repeat_interleave(n_cal * (S - 1))


def set_calib(n):
    """Use the first n calibration sequences per language; the n-th is held out to check every fit."""
    global NCAL, ROWS, HOLD, FIT
    NCAL, ROWS = n, SEQ_ID < n
    HOLD = SEQ_ID[ROWS] == n - 1
    FIT = ~HOLD


set_calib(n_cal)
hs = lambda i: HS[i][ROWS]


def ridge(X, Y, lam=None):
    """argmin_W ||X W - Y||^2 + lam*mean(diag(X^T X))*||W||^2 in float64."""
    dev = DEV if DEV == "cuda" else "cpu"  # MPS has no float64
    X, Y = X.to(dev, torch.float64), Y.to(dev, torch.float64)
    G = X.T @ X
    lam = CFG["ridge"] if lam is None else lam
    return torch.linalg.solve(G + lam * G.trace() / len(G) * torch.eye(len(G), dtype=G.dtype, device=dev), X.T @ Y).float().cpu()


def rel_err(P, T):
    """Held-out relative mismatch to the original model's hidden state."""
    return ((P[HOLD] - T[HOLD]).norm() / T[HOLD].norm()).item()


def angular(a, b):
    return (torch.arccos(F.cosine_similarity(a, b, -1).clamp(-1, 1)).mean() / math.pi).item()


def plan_pairs(k):
    """k non-overlapping adjacent pairs whose input and output states are most similar (Gromov et al. angular distance)."""
    p = CFG["protect"]
    score = {i: angular(HS[i], HS[i + 2]) for i in range(p, L - p - 1)}
    chosen = []
    for i in sorted(score, key=score.get):
        if all(abs(i - j) > 1 for j in chosen):
            chosen.append(i)
        if len(chosen) == k:
            break
    return [[i, i + 1] for i in sorted(chosen)]


def with_layers(layers):
    """Shallow copy of the original model that shares everything except the layer list."""
    s = copy.copy(teacher)
    s._modules = dict(teacher._modules)
    m = copy.copy(teacher.model)
    m._modules = dict(teacher.model._modules)
    m.layers = nn.ModuleList(layers)
    s.model = m
    return s


def assemble(plan, merged):
    """Each group of original layers in plan is replaced by one layer (merged[gi])."""
    layers, i, gi = [], 0, 0
    while i < L:
        if gi < len(plan) and i == plan[gi][0]:
            layers.append(merged[gi])
            i, gi = plan[gi][-1] + 1, gi + 1
        else:
            layers.append(teacher.model.layers[i])
            i += 1
    return with_layers(layers)


def student_index(plan, gi):
    return plan[gi][0] - sum(len(g) - 1 for g in plan[:gi])


print(f"{SLUG}: {L} layers, {len(HS[0])} calibration tokens ({int(HOLD.sum())} held out)")

In [ ]:
REF_PATH = f"{CFG['data_dir']}/results_reference_{SLUG}.json"  # original + drop rows, shared by all notebooks
os.makedirs(CFG["data_dir"], exist_ok=True)
load_json = lambda p: json.load(open(p)) if os.path.exists(p) else {}
pstr = lambda plan: "; ".join(f"{g[0]}-{g[-1]}" for g in plan) or "—"
done = lambda path, method, k, plan: f"{method}|{k}|{pstr(plan)}" in load_json(path)


def record(path, method, k, plan, model, build_s=0.0):
    t0 = time.time()
    e = evaluate(model)
    R = load_json(path)
    R[f"{method}|{k}|{pstr(plan)}"] = dict(method=method, k=k, plan=pstr(plan), layers=len(model.model.layers),
                                           params=sum(p.numel() for p in model.parameters()), build_s=round(build_s, 1), eval=e)
    json.dump(R, open(path, "w"), indent=1)
    print(f"{method} k={k} [{pstr(plan)}]: eval {time.time() - t0:.0f}s", flush=True)


def run_all(algo, build, plans=None):
    """Original + `drop` reference (keep the first layer of each group) + the algorithm, for every merge count."""
    res = f"{CFG['data_dir']}/results_{algo}_{SLUG}.json"
    if not done(REF_PATH, "original", 0, []):
        record(REF_PATH, "original", 0, [], teacher)
    for k in CFG["merge_counts"]:
        plan = plans[k] if plans else plan_pairs(k)
        print(f"\n=== {k} layers removed: groups {pstr(plan)} ===", flush=True)
        if not done(REF_PATH, "drop", k, plan):
            record(REF_PATH, "drop", k, plan, assemble(plan, [teacher.model.layers[g[0]] for g in plan]))
        if not done(res, algo, k, plan):
            t0 = time.time()
            m = build(plan)
            record(res, algo, k, plan, m, time.time() - t0)
            del m
            gc.collect()
            if DEV == "cuda":
                torch.cuda.empty_cache()
    return res


def table(paths=None):
    """One row per (method, layers removed, plan). PPL: mean % increase over languages (and the worst language).
    Accuracy: mean over languages, in %. `avg acc` = mean of the five few-shot benchmarks."""
    paths = paths or sorted(glob.glob(f"{CFG['data_dir']}/results_*_{SLUG}.json"))
    R = {}
    for p in paths:
        R.update(load_json(p))
    o = R["original|0|—"]
    rows = []
    for r in R.values():
        e, row = r["eval"], dict(method=r["method"], removed=r["k"], plan=r["plan"],
                                 **{"param red %": round(100 * (1 - r["params"] / o["params"]), 1)})
        for ds in ("mc4", "wiki"):
            if e[ds]:
                inc = [100 * (e[ds][l] / o["eval"][ds][l] - 1) for l in e[ds]]
                row[f"{ds} ppl +%"] = round(sum(inc) / len(inc), 1)
                row[f"{ds} worst +%"] = round(max(inc), 1)
        accs = []
        for ds in TASKS:
            if e.get(ds):
                row[ds] = round(100 * sum(e[ds].values()) / len(e[ds]), 1)
                accs.append(row[ds])
        row["avg acc"] = round(sum(accs) / max(1, len(accs)), 1)
        row["build s"] = r["build_s"]
        rows.append(row)
    df = pd.DataFrame(rows).sort_values(["removed", "avg acc"], ascending=[True, False]).reset_index(drop=True)
    pd.set_option("display.width", 250)
    pd.set_option("display.max_columns", 30)
    return df

## PRISM library: exact parallel form, selection, refit, proof-of-merge report

In [ ]:
# PRISM: Parallel-Residual Interaction-Sum Merge.  Shared by notebooks 07-10.
C_ = teacher.config
H, KV, HD, IM = C_.num_attention_heads, C_.num_key_value_heads, C_.head_dim, C_.intermediate_size
REP = H // KV
FD = DEV if DEV == "cuda" else "cpu"  # float64 linear algebra (MPS has none)
W = lambda m: m.weight.detach().float()


def cap0(model, li, inputs=(), outputs=()):
    """capture() keeping position 0 (the attention sink). Rows: language, sequence, position."""
    global flat
    old = flat
    flat = lambda t: (t[0] if isinstance(t, tuple) else t).flatten(0, -2).float().cpu()
    try:
        return capture(model, li, inputs, outputs)
    finally:
        flat = old


def masks0():
    seq = torch.arange(NCAL).repeat_interleave(S).repeat(len(LANGS))
    pos = torch.arange(S).repeat(NCAL * len(LANGS))
    return seq < NCAL - 1, seq == NCAL - 1, pos == 0, torch.arange(len(LANGS)).repeat_interleave(NCAL * S)


def tok_w(T):
    """Per-token weight. sink_aware: 1/||target||^2, so every token's RELATIVE error counts equally and the huge-norm
    attention-sink token is fitted too (not ignored, not dominating). Otherwise: position 0 left out (notebooks 01-06)."""
    fit, _, sink, _ = masks0()
    if CFG["sink_aware"]:
        w = 1 / T.pow(2).mean(1).clamp(min=1e-8)
        return w / w[fit].mean()
    return (~sink).float()


def wfit(X, Y, w):
    fit = masks0()[0] & (w > 0)
    s = w[fit].sqrt()[:, None]
    return ridge(X[fit] * s, Y[fit] * s)


def terr(P, T):
    """Held-out RMS relative error per token: (body tokens, attention-sink tokens)."""
    _, hold, sink, _ = masks0()
    e = lambda r: ((P[r] - T[r]).pow(2).sum(1) / T[r].pow(2).sum(1)).mean().sqrt().item()
    return e(hold & ~sink), e(hold & sink)


def setw(lin, w):
    lin.weight = nn.Parameter(w.contiguous().to(DEV), requires_grad=False)
    lin.out_features, lin.in_features = w.shape


def prism_layer(gsel, nsel):
    """ONE decoder layer made of parts of several original layers, computed in PARALLEL from one shared input.
    gsel: [(layer, kv_group)]: each KV group brings its REP query heads, its k/v rows, its o_proj columns and its own
    q_norm / k_norm gains. nsel: [(layer, neuron)]: gate/up rows + down_proj column. Each source layer's input
    RMSNorm gains are folded exactly into its read weights (W diag(gamma)), so the shared norms become 1."""
    Lr = teacher.model.layers
    new = copy.deepcopy(Lr[gsel[0][0]])
    sa, mlp = new.self_attn, new.mlp
    qr = lambda g: slice(g * REP * HD, (g + 1) * REP * HD)
    kr = lambda g: slice(g * HD, (g + 1) * HD)
    gi = {j: W(Lr[j].input_layernorm) for j, _ in gsel}
    gp = {j: W(Lr[j].post_attention_layernorm) for j, _ in nsel}
    setw(sa.q_proj, torch.cat([W(Lr[j].self_attn.q_proj)[qr(g)] * gi[j] for j, g in gsel]))
    setw(sa.k_proj, torch.cat([W(Lr[j].self_attn.k_proj)[kr(g)] * gi[j] for j, g in gsel]))
    setw(sa.v_proj, torch.cat([W(Lr[j].self_attn.v_proj)[kr(g)] * gi[j] for j, g in gsel]))
    setw(sa.o_proj, torch.cat([W(Lr[j].self_attn.o_proj)[:, qr(g)] for j, g in gsel], 1))
    sa.q_norm.weight = nn.Parameter(torch.stack([W(Lr[j].self_attn.q_norm) for j, _ in gsel for _ in range(REP)]), requires_grad=False)
    sa.k_norm.weight = nn.Parameter(torch.stack([W(Lr[j].self_attn.k_norm) for j, _ in gsel]), requires_grad=False)
    srcs = sorted({j for j, _ in nsel})
    idx = {j: torch.tensor([n for i, n in nsel if i == j], device=DEV) for j in srcs}
    for name, g in (("gate_proj", gp), ("up_proj", gp)):
        setw(getattr(mlp, name), torch.cat([W(getattr(Lr[j].mlp, name))[idx[j]] * g[j] for j in srcs]))
    setw(mlp.down_proj, torch.cat([W(Lr[j].mlp.down_proj)[:, idx[j]] for j in srcs], 1))
    for n in (new.input_layernorm, new.post_attention_layernorm):
        n.weight.data.fill_(1.0)
    return new


def _selfcheck():  # all parts of one layer -> must reproduce that layer exactly (the construction is lossless)
    global AMP_DTYPE
    j = L // 2
    m = prism_layer([(j, g) for g in range(KV)], [(j, n) for n in range(IM)])
    amp_, AMP_DTYPE = AMP_DTYPE, None  # fp32: half-precision rounding (~3e-4) would hide a real construction error
    try:
        with torch.no_grad():
            a = capture(teacher, j, [], [""])[":out"][:300]
            b = capture(assemble([[j]], [m]), j, [], [""])[":out"][:300]
    finally:
        AMP_DTYPE = amp_
    assert (a - b).norm() / a.norm() < 1e-4, (a - b).norm() / a.norm()


_selfcheck()


def components(j, w):
    """Per-token write energy of every KV group and every neuron of original layer j (in its own context)."""
    lay = teacher.model.layers[j]
    c = cap0(teacher, j, ["self_attn.o_proj", "mlp.down_proj"], ["self_attn"])
    Z, A = c["self_attn.o_proj"], c["mlp.down_proj"]
    Wo, Wd = W(lay.self_attn.o_proj).cpu(), W(lay.mlp.down_proj).cpu()
    ge = torch.stack([(Z[:, qr] @ Wo[:, qr].T).pow(2).sum(1) for qr in
                      (slice(g * REP * HD, (g + 1) * REP * HD) for g in range(KV))], 1)
    ne = A.pow(2) * Wd.pow(2).sum(0)
    return dict(Z=Z, A=A, ge=ge, ne=ne, wn=Wd.norm(dim=0), write=c["self_attn:out"])


def pivoted_cholesky(G, k):
    """Greedy column-subset selection: each step takes the column with the most energy NOT explained by the ones
    already taken (so redundant neurons, e.g. near-duplicates across the two layers, are not both kept)."""
    d, Lm, piv = G.diag().clone(), torch.zeros(len(G), k, dtype=G.dtype, device=G.device), []
    for m in range(k):
        i = int(torch.argmax(d))
        piv.append(i)
        l = (G[:, i] - Lm[:, :m] @ Lm[i, :m]) / d[i].clamp(min=1e-30).sqrt()
        Lm[:, m] = l
        d = d - l * l
        d[piv] = -float("inf")
    return torch.tensor(piv)


def select(comps, g, w, mode):
    """Choose KV groups and neurons for the merged layer out of ALL layers in g."""
    fit, _, _, lang = masks0()
    GE = torch.cat([comps[j]["ge"] for j in g], 1)
    NE = torch.cat([comps[j]["ne"] for j in g], 1)
    gid = [(j, x) for j in g for x in range(KV)]
    nid = [(j, x) for j in g for x in range(IM)]
    if mode in ("energy", "polyglot"):
        def score(E):
            if mode == "energy":
                return (E[fit] * w[fit, None]).sum(0)
            shares = []
            for li in range(len(LANGS)):  # each language's share of the energy; keep what ANY language relies on
                r = fit & (lang == li)
                e = (E[r] * w[r, None]).sum(0)
                shares.append(e / e.sum())
            return torch.stack(shares).max(0).values
        gs, ns = score(GE).topk(KV).indices, score(NE).topk(IM).indices
    else:  # span
        s = w[fit].sqrt()[:, None]
        Z = torch.cat([comps[j]["Z"][fit] for j in g], 1) * s
        Y = sum(comps[j]["write"][fit] for j in g) * s
        Zd, Yd = Z.to(FD), Y.to(FD)
        G, C = (Zd.T @ Zd).double(), (Zd.T @ Yd).double()
        lam = CFG["ridge"] * G.trace() / len(G)
        blk = [torch.arange(b * REP * HD, (b + 1) * REP * HD, device=G.device) for b in range(len(gid))]
        chosen = []
        for _ in range(KV):  # greedy: the KV group that most improves the best linear reconstruction of ALL writes
            best = None
            for b in set(range(len(gid))) - set(chosen):
                ix = torch.cat([blk[c] for c in chosen + [b]])
                Cs = C[ix]
                v = (Cs * torch.linalg.solve(G[ix][:, ix] + lam * torch.eye(len(ix), dtype=G.dtype, device=G.device), Cs)).sum()
                if best is None or v > best[1]:
                    best = (b, v)
            chosen.append(best[0])
        gs = torch.tensor(chosen)
        A = torch.cat([comps[j]["A"][fit] * comps[j]["wn"] for j in g], 1) * s
        Ad = A.to(FD)
        ns = pivoted_cholesky((Ad.T @ Ad).double(), IM)
        del Z, Y, Zd, Yd, G, C, A, Ad
    return sorted(gid[i] for i in gs.tolist()), sorted(nid[i] for i in ns.tolist())


def knockout(layer, gsel, nsel, keep_from):
    """Copy of the merged layer with the writes of every part NOT from layer `keep_from` zeroed."""
    m = copy.deepcopy(layer)
    gmask = torch.tensor([j == keep_from for j, _ in gsel for _ in range(REP * HD)], device=DEV)
    order = [j for j in sorted({j for j, _ in nsel}) for i, _ in nsel if i == j]
    nmask = torch.tensor([j == keep_from for j in order], device=DEV)
    m.self_attn.o_proj.weight.data *= gmask
    m.mlp.down_proj.weight.data *= nmask
    return m


def residual_commonality(g):
    """How alike the inputs of the layers in g are (cosine of RMS-normalized hidden states, non-sink tokens)."""
    return min(F.cosine_similarity(HS[g[0]], HS[j], -1).mean().item() for j in g[1:])


def projection_commonality(r=64):
    """For every adjacent pair: overlap of the top-r WRITE subspaces ([o_proj | down_proj] column space) and READ
    subspaces ([q;k;v;gate;up] diag(gamma) row space). 1 = same subspace, r/d = random."""
    U, V = [], []
    for lay in teacher.model.layers:
        sa, ml = lay.self_attn, lay.mlp
        Wr = torch.cat([W(sa.o_proj), W(ml.down_proj)], 1).to(FD)
        Rd = torch.cat([torch.cat([W(sa.q_proj), W(sa.k_proj), W(sa.v_proj)]) * W(lay.input_layernorm),
                        torch.cat([W(ml.gate_proj), W(ml.up_proj)]) * W(lay.post_attention_layernorm)]).to(FD)
        U.append(torch.linalg.svd(Wr, full_matrices=False)[0][:, :r])
        V.append(torch.linalg.svd(Rd, full_matrices=False)[2][:r].T)
    ov = lambda P, Q: ((P.T @ Q).pow(2).sum() / r).item()
    return pd.DataFrame([dict(pair=f"{i}-{i + 1}", angular=round(angular(HS[i], HS[i + 2]), 3),
                              input_cos=round(residual_commonality([i, i + 1]), 3),
                              write_overlap=round(ov(U[i], U[i + 1]), 3), read_overlap=round(ov(V[i], V[i + 1]), 3))
                         for i in range(L - 2)])  # the last pair has no stored state after it


def prism_build(plan, mode, refit=True):
    """Merge every group in forward order inside one student (later groups see earlier groups' errors)."""
    student = assemble(plan, [teacher.model.layers[g[0]] for g in plan])
    for gi, g in enumerate(plan):
        si = student_index(plan, gi)
        T = cap0(teacher, g[-1], [], [""])[":out"]  # original state after the group (incl. sink)
        w = tok_w(T)
        comps = {j: components(j, w) for j in g}
        err = lambda layer: (student.model.layers.__setitem__(si, layer), terr(cap0(student, si, [], [""])[":out"], T))[1]
        rep = {f"drop (keep L{j})": err(teacher.model.layers[j]) for j in g}
        rep["parallel, all parts (|g|x size)"] = err(prism_layer([(j, x) for j in g for x in range(KV)],
                                                                   [(j, x) for j in g for x in range(IM)]))
        gsel, nsel = select(comps, g, w, mode)
        layer = prism_layer(gsel, nsel)
        rep[f"PRISM-{mode}, no refit"] = err(layer)
        if refit:  # absorb what the sequential interaction and the discarded parts wrote, by least squares
            writes = sum(comps[j]["write"] for j in g)
            Z = cap0(student, si, ["self_attn.o_proj"])["self_attn.o_proj"]
            setw(layer.self_attn.o_proj, wfit(Z, writes, w).T)
            c = cap0(student, si, ["mlp.down_proj"], [""])
            D, Y = c["mlp.down_proj"], c[":out"]
            setw(layer.mlp.down_proj, wfit(D, T - Y + D @ W(layer.mlp.down_proj).cpu().T, w).T)
            rep[f"PRISM-{mode} + refit"] = err(layer)
        for j in g:
            rep[f"   knockout: only L{j}'s parts kept"] = err(knockout(layer, gsel, nsel, j))
        student.model.layers[si] = layer
        print(f"\n  group {pstr([g])}: input cosine {residual_commonality(g):.3f};  merged layer takes "
              + ", ".join(f"L{j}: {sum(x == j for x, _ in gsel)} KV groups + {sum(x == j for x, _ in nsel)} neurons" for j in g))
        print("    held-out relative error vs original state after the group (body tokens | sink token):")
        for k_, (b_, s_) in rep.items():
            print(f"    {k_:<38s} {b_:7.1%} | {s_:7.1%}")
        del comps
        gc.collect()
    extra = sum(p.numel() for n, p in student.named_parameters() if "q_norm" in n or "k_norm" in n) - \
        sum(p.numel() for n, p in assemble(plan, [teacher.model.layers[g[0]] for g in plan]).named_parameters() if "q_norm" in n or "k_norm" in n)
    print(f"  merged model: {len(student.model.layers)} layers (original {L}); {sum(p.numel() for p in student.parameters()):,} params "
          f"(+{extra:,} from per-head q/k-norm gains, {100 * extra / sum(p.numel() for p in teacher.parameters()):.4f} %)")
    return student


variant = lambda mode: f"prism_{mode}" + ("" if CFG["sink_aware"] else "_nosink") + ("" if CFG["refit"] else "_norefit")

## Shared distillation code

In [ ]:
ALPHA = {"v": 1.0}  # 1 = original group, 0 = merged layer only (used by Fade / Blend)
TRAIN_DTYPE = AMP_DTYPE


def outputs_of(model, idx):
    got = {}
    return got, [model.model.layers[i].register_forward_hook(lambda m, a, o, i=i: got.__setitem__(i, o)) for i in idx]


def kd_losses(student, x, t_out, s_out, pairs):
    ctx = lambda: torch.autocast("cuda", dtype=TRAIN_DTYPE) if TRAIN_DTYPE else contextlib.nullcontext()
    with torch.no_grad(), ctx():
        th = teacher.model(x).last_hidden_state
    with ctx():
        sh = student.model(x).last_hidden_state
    hid = sum((s_out[si][:, 1:].float() - t_out[ti][:, 1:].float()).pow(2).mean() / t_out[ti][:, 1:].float().pow(2).mean()
              for si, ti in pairs) / len(pairs)
    kl, d = 0.0, th.shape[-1]
    for a, b in zip(sh.reshape(-1, d).split(512), th.reshape(-1, d).split(512)):  # chunks: full-vocab logits are big
        with ctx():
            ls, lt = teacher.lm_head(a), teacher.lm_head(b)
        kl = kl + F.kl_div(F.log_softmax(ls.float(), -1), F.log_softmax(lt.float(), -1), log_target=True, reduction="sum")
    return kl / x.numel(), hid


def kd_train(student, pairs, params, batch_fn=None):
    """Language-balanced distillation (one sequence per language per step): KL(original || student) on next-token
    distributions + normalized hidden-state MSE at each (student layer, original layer) pair. ALPHA follows a cosine
    from 1 to 0 over the first `anneal_frac` of the steps, then stays 0."""
    t_out, th = outputs_of(teacher, [t for _, t in pairs])
    s_out, sh = outputs_of(student, [s for s, _ in pairs])
    opt = torch.optim.AdamW(params, lr=CFG["lr"], weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda", enabled=TRAIN_DTYPE == torch.float16)
    N, A = CFG["steps"], int(CFG["anneal_frac"] * CFG["steps"])
    gen = torch.Generator().manual_seed(0)
    batch = batch_fn or (lambda: torch.stack([DATA[l]["train"][torch.randint(len(DATA[l]["train"]), (1,), generator=gen)[0]]
                                              for l in LANGS]).to(DEV))
    try:
        ALPHA["v"] = 1.0
        with torch.no_grad():
            kl, hid = kd_losses(student, batch(), t_out, s_out, pairs)
        assert kl.item() < 1e-3 and hid.item() < 1e-3, f"at alpha=1 the student must equal the original (kl {kl}, hid {hid})"
        t0 = time.time()
        for step in range(N):
            ALPHA["v"] = 0.5 * (1 + math.cos(math.pi * (step + 1) / A)) if step + 1 < A else 0.0  # < 1 from step 0: the merged layer gets gradient
            x = batch()
            opt.zero_grad(set_to_none=True)
            for xb in x.split(max(1, 1536 // x.shape[1])):  # micro-batches of <= 1536 tokens (fits a T4); same gradient
                kl, hid = kd_losses(student, xb, t_out, s_out, pairs)
                scaler.scale((kl + CFG["hid_w"] * hid) * len(xb) / len(x)).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            if step % 50 == 0 or step == N - 1:
                print(f"  step {step:4d}  alpha {ALPHA['v']:.3f}  kl {kl.item():.4f}  hid {hid.item():.4f}  ({time.time() - t0:.0f}s)",
                      flush=True)
    finally:
        for h in th + sh:
            h.remove()
        ALPHA["v"] = 0.0

## Shrink library: parallel union annealed to one layer, ICL packing, language-specific residual, CoPlan

In [ ]:
class LangDelta(nn.Module):
    """Language-specific low-rank residual beside a merged layer (LSR):
    delta(h) = sum_l p_l(h) * B_l A_l rms(h), with p = a frozen linear LANGUAGE PROBE on the layer input (soft, per
    token, so mixed-language text works). The shared merged layer carries what the pair computes for every language;
    the small per-language part carries what merging loses for one language. B starts at 0: no change at init."""

    def __init__(self, probe_w, probe_b, rank):
        super().__init__()
        nl, d = probe_w.shape
        self.register_buffer("pw", probe_w)
        self.register_buffer("pb", probe_b)
        self.A = nn.Parameter(torch.randn(nl, rank, d, device=DEV) / d ** 0.5)
        self.B = nn.Parameter(torch.zeros(nl, d, rank, device=DEV))

    def forward(self, h):
        x = h.float() * torch.rsqrt(h.float().pow(2).mean(-1, keepdim=True) + 1e-6)
        p = torch.softmax(x @ self.pw.T + self.pb, -1)
        z = torch.einsum("...d,lrd->...lr", x, self.A.float()) * p[..., None]
        return torch.einsum("...lr,ldr->...d", z, self.B.float()).to(h.dtype)


class LSRLayer(nn.Module):
    """A merged standard Qwen3 layer + its language-specific low-rank residual."""

    def __init__(self, inner, delta):
        super().__init__()
        self.inner, self.delta = inner, delta

    def forward(self, h, *a, **k):
        return self.inner(h, *a, **k) + self.delta(h)


def fit_probe(li):
    """Linear language probe on the RMS-normalized state entering layer li (calibration tokens, held-out accuracy)."""
    X, y = hs(li), TOK_LANG[ROWS]
    X = X * torch.rsqrt(X.pow(2).mean(-1, keepdim=True) + 1e-6)
    dev = DEV if DEV == "cuda" else "cpu"
    Xd, yd = X.to(dev), y.to(dev)
    lin = nn.Linear(X.shape[1], len(LANGS)).to(dev)
    opt = torch.optim.Adam(lin.parameters(), lr=1e-2)
    with torch.enable_grad():
        for _ in range(300):
            loss = F.cross_entropy(lin(Xd[FIT.to(dev)]), yd[FIT.to(dev)])
            opt.zero_grad()
            loss.backward()
            opt.step()
    with torch.no_grad():
        acc = (lin(Xd[HOLD.to(dev)]).argmax(-1) == yd[HOLD.to(dev)]).float().mean().item()
    return lin.weight.detach().to(DEV), lin.bias.detach().to(DEV), acc


def packed_batch_fn():
    """ICL-preserving distillation text. Each sequence = [a, b, a]: a chunk of the anchor language, a chunk of ANOTHER
    language, then the first chunk AGAIN. Predicting the repeat needs long-range copying (the induction / in-context
    circuits few-shot prompts rely on), and the switch trains cross-lingual context. Plain 256-token Wiki chunks
    exercise neither; that is why merges that look fine on perplexity lose few-shot accuracy."""
    gen = torch.Generator().manual_seed(0)
    pick = lambda l: DATA[l]["train"][torch.randint(len(DATA[l]["train"]), (1,), generator=gen)[0]]

    def batch():
        seqs = []
        for i, l in enumerate(LANGS):
            o = LANGS[(i + 1 + int(torch.randint(len(LANGS) - 1, (1,), generator=gen))) % len(LANGS)]
            a = pick(l)
            seqs.append(torch.cat([a, pick(o), a]))
        return torch.stack(seqs).to(DEV)
    return batch


def coplan(counts):
    """CoPlan: choose pairs by PARALLELIZABILITY = held-out error of the exact parallel union of each adjacent pair,
    measured inside the original model. PRISM merges through the parallel form, so pairs where it is most exact
    (residual commonality) are merged first, instead of pairs that merely change the state least (angular distance)."""
    p, err = CFG["protect"], {}
    for i in range(p, L - p - 1):
        g = [i, i + 1]
        U = prism_layer([(j, x) for j in g for x in range(KV)], [(j, x) for j in g for x in range(IM)])
        err[i] = terr(cap0(assemble([g], [U]), i, [], [""])[":out"], cap0(teacher, i + 1, [], [""])[":out"])[0]
        del U
    print("  parallel-union error per pair: " + ", ".join(f"{i}-{i + 1} {e:.1%}" for i, e in err.items()))
    plans = {}
    for k in counts:
        chosen = []
        for i in sorted(err, key=err.get):
            if all(abs(i - j) > 1 for j in chosen):
                chosen.append(i)
            if len(chosen) == k:
                break
        plans[k] = [[i, i + 1] for i in sorted(chosen)]
        print(f"  CoPlan, {k} layers removed: {pstr(plans[k])}   (angular-distance plan: {pstr(plan_pairs(k))})")
    return plans


class Shrink(nn.Module):
    """alpha * (original group, run sequentially) + (1 - alpha) * [U(h) + delta(h)], where U = the exact PARALLEL union
    of the group whose to-be-discarded parts are scaled by alpha, and delta = optional LSR.
    alpha = 1: exactly the original model.  alpha = 0: only the kept parts (+ LSR) remain = one standard-size layer."""

    def __init__(self, group, union, gmask, nmask, delta=None):
        super().__init__()
        self.group, self.union, self.delta = nn.ModuleList(group), union, delta
        self.register_buffer("gm", gmask.float().to(DEV))
        self.register_buffer("nm", nmask.float().to(DEV))
        fade = lambda m: lambda mod, a: (a[0] * (m + ALPHA["v"] * (1 - m)).to(a[0].dtype),)
        union.self_attn.o_proj.register_forward_pre_hook(fade(self.gm))
        union.mlp.down_proj.register_forward_pre_hook(fade(self.nm))

    def forward(self, h, *a, **k):
        al = ALPHA["v"]
        y = h
        if al > 0:
            for m in self.group:
                y = m(y, *a, **k)
            if al == 1:
                return y
        u = self.union(h, *a, **k)
        if self.delta is not None:
            u = u + self.delta(h)
        return u if al == 0 else al * y + (1 - al) * u


def extract(union, gid, gsel, nid, nsel):
    """The kept parts of the trained union, as a normal Qwen3 layer (8 KV groups, 3072 neurons)."""
    new, ua = copy.deepcopy(union), union.self_attn
    for hk in list(new.self_attn.o_proj._forward_pre_hooks) + list(new.mlp.down_proj._forward_pre_hooks):
        new.self_attn.o_proj._forward_pre_hooks.pop(hk, None)
        new.mlp.down_proj._forward_pre_hooks.pop(hk, None)
    sa = new.self_attn
    gi = [gid.index(t) for t in gsel]
    ni = torch.tensor([nid.index(t) for t in nsel], device=DEV)
    qi = torch.cat([torch.arange(b * REP * HD, (b + 1) * REP * HD) for b in gi]).to(DEV)
    ki = torch.cat([torch.arange(b * HD, (b + 1) * HD) for b in gi]).to(DEV)
    setw(sa.q_proj, W(ua.q_proj)[qi])
    setw(sa.k_proj, W(ua.k_proj)[ki])
    setw(sa.v_proj, W(ua.v_proj)[ki])
    setw(sa.o_proj, W(ua.o_proj)[:, qi])
    sa.q_norm.weight = nn.Parameter(W(ua.q_norm)[torch.tensor([b * REP + r for b in gi for r in range(REP)], device=DEV)], requires_grad=False)
    sa.k_norm.weight = nn.Parameter(W(ua.k_norm)[torch.tensor(gi, device=DEV)], requires_grad=False)
    for name in ("gate_proj", "up_proj"):
        setw(getattr(new.mlp, name), W(getattr(union.mlp, name))[ni])
    setw(new.mlp.down_proj, W(union.mlp.down_proj)[:, ni])
    return new.requires_grad_(False)


def shrink_build(plan, select_mode, pack=False, lsr_rank=0):
    """PRISM-Shrink and its extensions: + ICL-preserving packed distillation (pack) and + language-specific residual (lsr_rank > 0)."""
    shr, info = [], []
    for g in plan:
        T = cap0(teacher, g[-1], [], [""])[":out"]
        w = tok_w(T)
        gsel, nsel = select({j: components(j, w) for j in g}, g, w, select_mode)
        gid, nid = [(j, x) for j in g for x in range(KV)], [(j, x) for j in g for x in range(IM)]
        union = prism_layer(gid, nid).requires_grad_(True)
        gm = torch.tensor([t in set(gsel) for t in gid]).repeat_interleave(REP * HD)
        nm = torch.tensor([t in set(nsel) for t in nid])
        delta, note = None, ""
        if lsr_rank:
            pw, pb, acc = fit_probe(g[0])
            delta, note = LangDelta(pw, pb, lsr_rank), f"; language probe on its input: {acc:.1%} held-out accuracy"
        shr.append(Shrink([teacher.model.layers[j] for j in g], union, gm, nm, delta))
        info.append((gid, gsel, nid, nsel))
        print(f"  group {pstr([g])}: input cosine {residual_commonality(g):.3f}; kept "
              + ", ".join(f"L{j}: {sum(x == j for x, _ in gsel)} KV groups + {sum(x == j for x, _ in nsel)} neurons" for j in g)
              + note, flush=True)
    student = assemble(plan, shr)
    pairs = [(student_index(plan, gi), g[-1]) for gi, g in enumerate(plan)]
    params = [p for s in shr for p in s.union.parameters()] + [p for s in shr if s.delta is not None for p in s.delta.parameters()]
    kd_train(student, pairs, params, packed_batch_fn() if pack else None)
    merged = []
    for s, inf in zip(shr, info):
        layer = extract(s.union, *inf)
        merged.append(layer if s.delta is None else LSRLayer(layer, s.delta.requires_grad_(False)))
    out = assemble(plan, merged)
    for gi, g in enumerate(plan):  # proof: the extracted layer == the trained union at alpha = 0
        si = student_index(plan, gi)
        T = cap0(teacher, g[-1], [], [""])[":out"]
        e_union, e_final = terr(cap0(student, si, [], [""])[":out"], T)[0], terr(cap0(out, si, [], [""])[":out"], T)[0]
        print(f"  group {pstr([g])}: held-out error, trained union at alpha=0 {e_union:.1%} = extracted layer {e_final:.1%}")
    n_lsr = sum(p.numel() for s in shr if s.delta is not None for p in s.delta.parameters())
    print(f"  merged model: {len(out.model.layers)} layers (original {L}); {sum(p.numel() for p in out.parameters()):,} params"
          + (f" (incl. {n_lsr:,} LSR = {100 * n_lsr / sum(p.numel() for p in teacher.parameters()):.2f} % of the original)" if n_lsr else ""))
    return out

## Algorithm: PolyShrink (CoPlan + Shrink + ICL packing + LSR)

In [ ]:
ALGO = f"polyshrink_full_lsr{CFG['lsr_rank']}_{CFG['select']}{CFG['anneal_frac']}"
PLANS = coplan(CFG["merge_counts"])  # which pairs: parallelizability, not angular distance
build = lambda plan: shrink_build(plan, CFG["select"], pack=True, lsr_rank=CFG["lsr_rank"])

## Run and results
Finished runs are saved and skipped on re-run.

In [ ]:
RES = run_all(ALGO, build, globals().get("PLANS"))  # PLANS: searched groups (notebook 04); else angular-distance pairs
print("This algorithm vs the references:")
display(table([REF_PATH, RES]))
print("Every algorithm whose results are in", CFG["data_dir"], "(run the other notebooks with the same data_dir):")
display(table())

## Kaggle run log (sumitzha/layer-merging-15-polyshrink-full, GPU T4 x2)
Everything the notebook printed during the Kaggle run, in order. The executed notebook with outputs in place: https://www.kaggle.com/code/sumitzha/layer-merging-15-polyshrink-full (File -> Download notebook).

In [ ]:
# output of the Kaggle run (not re-run)

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.














/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' cur